In [62]:
import glob
import os
import numpy as np
from pathlib import Path
from unlzw3 import unlzw
import pandas as pd
import torchvision
import urllib.request, subprocess, tempfile, shutil
import zipfile
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer

In [63]:
def parse_libsvm_file(path, n_features=128):
    X = []

    with open(path, "r") as f:
        for line in f:
            parts = line.strip().split()

            # skip empty
            if len(parts) == 0:
                continue

            # feature vector
            vec = np.zeros(n_features, dtype=np.float32)

            # start from index 1 (skip label)
            for item in parts[1:]:
                try:
                    idx, val = item.split(":")
                    idx = int(idx) - 1  # convert to 0-based
                    if 0 <= idx < n_features:
                        vec[idx] = float(val)
                except:
                    continue

            X.append(vec)

    return np.array(X, dtype=np.float32)

In [64]:
def save_chunks(X_full, chunk_rows, out_dir):
    """X_full: (N, d) float32.  Saves X_{0000}.npy, X_{0001}.npy, ..."""
    N = X_full.shape[0]
    idx = 0
    for start in range(0, N, chunk_rows):
        block = X_full[start : start + chunk_rows].astype(np.float32)
        path  = os.path.join(out_dir, f"X_{idx:04d}.npy")
        np.save(path, block)
        print(f"  saved {path}  {block.shape}")
        idx += 1
    return idx   # number of files written

In [65]:
# OPTIONS: "arxiv" | "cifar10" | "isolet" | "yearprediction" | "gas_sensor" | "har" | "fashion_mnist" | "news20" | "synthetic"
DATASET = "isolet"

CHUNK_ROWS = 1000
OUT_DIR    = "datasets"
os.makedirs(OUT_DIR, exist_ok=True)


# ── CIFAR-10 ─────────────────────────────────────────────────────────────────
if DATASET == "cifar10":
    print("Downloading CIFAR-10 ...")
    train = torchvision.datasets.CIFAR10(root="./raw_data", train=True,  download=True)
    test  = torchvision.datasets.CIFAR10(root="./raw_data", train=False, download=True)

    X = np.vstack([
        np.array(train.data).reshape(50000, -1),
        np.array(test.data ).reshape(10000, -1),
    ]).astype(np.float32)                          
    # (60000, 3072)

    print(f"CIFAR-10 matrix: {X.shape}")
    os.makedirs(f'{OUT_DIR}/cifar10', exist_ok=True)
    n_files = save_chunks(X, CHUNK_ROWS, f'{OUT_DIR}/cifar10')
    print(f"Written {n_files} chunk files.")


# ── ISOLET ───────────────────────────────────────────────────────────────────
elif DATASET == "isolet":
    base = "https://archive.ics.uci.edu/ml/machine-learning-databases/isolet/"
    files = ["isolet1+2+3+4.data.Z", "isolet5.data.Z"]
    tmp   = tempfile.mkdtemp()

    dfs = []
    for fname in files:
        url      = base + fname
        gz_path  = os.path.join(tmp, fname)
        dat_path = gz_path[:-2]          # strip .Z

        print(f"Downloading {url} ...")
        urllib.request.urlretrieve(url, gz_path)

        print(f"Decompressing {fname} ...")
        with open(gz_path, "rb") as f:
            compressed = f.read()

        decompressed = unlzw(compressed)

        with open(dat_path, "wb") as f:
            f.write(decompressed)

        df = pd.read_csv(dat_path, header=None)
        dfs.append(df)
        print(f"  loaded {df.shape}")

    X_df = pd.concat(dfs, ignore_index=True)
    X    = X_df.iloc[:, :-1].values.astype(np.float32)  
    X   -= X.mean(axis=0)

    shutil.rmtree(tmp)
    print(f"ISOLET matrix: {X.shape}")
    os.makedirs(f'{OUT_DIR}/isolet', exist_ok=True)
    n_files = save_chunks(X, CHUNK_ROWS, f'{OUT_DIR}/isolet')
    print(f"Written {n_files} chunk files.")

elif DATASET == "arxiv":
    ### TODO
    pass


# ── YearPredictionMSD ────────────────────────────────────────────────────────
elif DATASET == "yearprediction":
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00203/YearPredictionMSD.txt.zip"
    raw_dir = "./raw_data/yearprediction"
    os.makedirs(raw_dir, exist_ok=True)

    zip_path = os.path.join(raw_dir, "YearPredictionMSD.txt.zip")
    txt_path = os.path.join(raw_dir, "YearPredictionMSD.txt")

    # Download
    if not os.path.exists(txt_path):
        if not os.path.exists(zip_path):
            print("Downloading YearPredictionMSD...")
            urllib.request.urlretrieve(url, zip_path)

        print("Extracting...")
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(raw_dir)

    print("Loading data (this may take a bit)...")

    # Load (it's CSV-like, no header)
    data = np.loadtxt(txt_path, delimiter=",")

    # First column is target → remove it
    X = data[:, 1:].astype(np.float32)

    # Optional: subset for your experiments
    MAX_SAMPLES = 100000
    if X.shape[0] > MAX_SAMPLES:
        print(f"Subsampling to {MAX_SAMPLES} samples...")
        X = X[:MAX_SAMPLES]

    print(f"YearPredictionMSD matrix: {X.shape}")

    os.makedirs(f"{OUT_DIR}/yearprediction", exist_ok=True)
    n_files = save_chunks(X, CHUNK_ROWS, f"{OUT_DIR}/yearprediction")

    print(f"Written {n_files} chunk files.")


# ── GAS SENSOR ARRAY DRIFT ───────────────────────────────────────────────────
elif DATASET == "gas_sensor":
    url = "https://archive.ics.uci.edu/static/public/224/gas+sensor+array+drift+dataset.zip"
    raw_dir = "./raw_data/gas_sensor"
    os.makedirs(raw_dir, exist_ok=True)

    zip_path = os.path.join(raw_dir, "gas_sensor.zip")

    # -----------------------
    # Download
    # -----------------------
    if not os.path.exists(zip_path):
        print("Downloading Gas Sensor Array Drift Dataset...")
        urllib.request.urlretrieve(url, zip_path)

    # -----------------------
    # Extract
    # -----------------------
    extract_dir = os.path.join(raw_dir, "extracted")
    if not os.path.exists(extract_dir):
        print("Extracting...")
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(extract_dir)

    # -----------------------
    # Parser (LIBSVM-style)
    # -----------------------
    def parse_libsvm_file(path, n_features=128):
        X = []

        with open(path, "r") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 0:
                    continue

                vec = np.zeros(n_features, dtype=np.float32)

                # skip label (parts[0])
                for item in parts[1:]:
                    try:
                        idx, val = item.split(":")
                        idx = int(idx) - 1  # 1-based → 0-based
                        if 0 <= idx < n_features:
                            vec[idx] = float(val)
                    except:
                        continue

                X.append(vec)

        return np.asarray(X, dtype=np.float32)

    # -----------------------
    # Load all batches
    # -----------------------
    files = sorted(glob.glob(os.path.join(extract_dir, "**/*.dat"), recursive=True))

    print(f"Found {len(files)} files")

    X_list = []
    for f in files:
        print(f"Parsing {f}")
        X_part = parse_libsvm_file(f, n_features=128)
        X_list.append(X_part)

    X = np.vstack(X_list).astype(np.float32)

    print(f"Raw matrix: {X.shape}")

    # -----------------------
    # Shuffle (important: drift ordering)
    # -----------------------
    rng = np.random.default_rng(0)
    # X = X[rng.permutation(X.shape[0])]

    print(f"Final matrix: {X.shape}")

    # -----------------------
    # Chunk saving
    # -----------------------
    os.makedirs(f"{OUT_DIR}/gas_sensor", exist_ok=True)
    n_files = save_chunks(X, CHUNK_ROWS, f"{OUT_DIR}/gas_sensor")

    print(f"Written {n_files} chunk files.")

# ── HUMAN ACTIVITY RECOGNITION ───────────────────────────────────────────────
elif DATASET == "har":
    url = (
        "https://archive.ics.uci.edu/ml/machine-learning-"
        "databases/00240/UCI%20HAR%20Dataset.zip"
    )

    raw_dir = "./raw_data/har"
    os.makedirs(raw_dir, exist_ok=True)

    zip_path = os.path.join(raw_dir, "har.zip")

    # -----------------------
    # Download
    # -----------------------
    if not os.path.exists(zip_path):
        print("Downloading HAR dataset...")
        urllib.request.urlretrieve(url, zip_path)

    # -----------------------
    # Extract
    # -----------------------
    extract_dir = os.path.join(raw_dir, "extracted")

    if not os.path.exists(extract_dir):
        print("Extracting...")
        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(extract_dir)

    # -----------------------
    # Load train/test
    # -----------------------
    base = os.path.join(
        extract_dir,
        "UCI HAR Dataset"
    )

    X_train = np.loadtxt(
        os.path.join(base, "train", "X_train.txt"),
        dtype=np.float32
    )

    X_test = np.loadtxt(
        os.path.join(base, "test", "X_test.txt"),
        dtype=np.float32
    )

    X = np.vstack([X_train, X_test])

    print(f"HAR matrix: {X.shape}")

    # -----------------------
    # Save chunks
    # -----------------------
    os.makedirs(f"{OUT_DIR}/har", exist_ok=True)

    n_files = save_chunks(
        X,
        CHUNK_ROWS,
        f"{OUT_DIR}/har"
    )

    print(f"Written {n_files} chunk files.")

# ── FASHION-MNIST ────────────────────────────────────────────────────────────
elif DATASET == "fashion_mnist":
    print("Downloading Fashion-MNIST...")

    train = torchvision.datasets.FashionMNIST(
        root="./raw_data",
        train=True,
        download=True
    )

    test = torchvision.datasets.FashionMNIST(
        root="./raw_data",
        train=False,
        download=True
    )

    # -----------------------
    # Flatten images
    # -----------------------
    X_train = train.data.numpy().reshape(len(train), -1)
    X_test  = test.data.numpy().reshape(len(test), -1)

    X = np.vstack([X_train, X_test]).astype(np.float32)

    print(f"Fashion-MNIST matrix: {X.shape}")

    # -----------------------
    # OPTIONAL:
    # normalize to [0,1]
    # -----------------------
    X /= 255.0

    # -----------------------
    # Save chunks
    # -----------------------
    os.makedirs(f"{OUT_DIR}/fashion_mnist", exist_ok=True)

    n_files = save_chunks(
        X,
        CHUNK_ROWS,
        f"{OUT_DIR}/fashion_mnist"
    )

    print(f"Written {n_files} chunk files.")

# ── NEWS20 ────────────────────────────────────────────────────────────────
elif DATASET == "news20":
    print("Downloading 20 Newsgroups...")

    dataset = fetch_20newsgroups(
        subset="all",
        remove=("headers", "footers", "quotes")
    )

    vectorizer = TfidfVectorizer(
        max_features=1000,
        stop_words="english",
        sublinear_tf=True,
        norm=None,
        dtype=np.float32
    )

    X_sparse = vectorizer.fit_transform(dataset.data)

    print(f"Sparse matrix: {X_sparse.shape}")

    # Convert to dense for consistent benchmarking
    X = X_sparse.toarray().astype(np.float32)

    # Optional centering
    # (recommended for PCA benchmarking)
    X -= X.mean(axis=0)

    print(f"Dense matrix: {X.shape}")

    os.makedirs(f"{OUT_DIR}/news20", exist_ok=True)

    n_files = save_chunks(
        X,
        CHUNK_ROWS,
        f"{OUT_DIR}/news20"
    )

    print(f"Written {n_files} chunk files.")


# ── SYNTHETIC ────────────────────────────────────────────────────────────────
elif DATASET == "synthetic":
    print("Generating Synthetic Low-Rank Data in chunks...")
    
    # Parameters
    N_SAMPLES  = 100000
    N_FEATURES = 256
    TRUE_RANK  = 30
    NOISE_STD  = 0.1
    
    rng = np.random.default_rng(42)
    
    # Create the fixed Right Singular Vectors (the "basis") 
    # This stays in memory but is small: (50 x 512)
    V_true = rng.standard_normal((TRUE_RANK, N_FEATURES), dtype=np.float32)
    
    os.makedirs(f"{OUT_DIR}/synthetic", exist_ok=True)
    
    total_files = 0
    # Process in CHUNK_ROWS to keep memory footprint tiny
    for i in range(0, N_SAMPLES, CHUNK_ROWS):
        actual_rows = min(CHUNK_ROWS, N_SAMPLES - i)
        
        # Generate random weights for this chunk: (CHUNK_ROWS x 50)
        U_chunk = rng.standard_normal((actual_rows, TRUE_RANK), dtype=np.float32)
        
        # Project into high-dimensional space: (CHUNK_ROWS x 512)
        X_chunk = U_chunk @ V_true
        
        # Add noise
        if NOISE_STD > 0:
            X_chunk += rng.normal(0, NOISE_STD, size=X_chunk.shape).astype(np.float32)
            
        # Standardize (approximate centering per chunk to save memory)
        X_chunk -= X_chunk.mean(axis=0)
        
        # Save this chunk directly to disk
        chunk_idx = i // CHUNK_ROWS
        filename = os.path.join(f"{OUT_DIR}/synthetic", f"X_{chunk_idx:04d}.npy")
        np.save(filename, X_chunk)
        total_files += 1

    print(f"Generated {N_SAMPLES} samples in {total_files} files.")
    print(f"Each chunk shape: ({CHUNK_ROWS}, {N_FEATURES})")

Decompressing isolet1+2+3+4.data.Z ...
  loaded (6238, 618)
Decompressing isolet5.data.Z ...
  loaded (1559, 618)
ISOLET matrix: (7797, 617)
  saved datasets/isolet\X_0000.npy  (1000, 617)
  saved datasets/isolet\X_0001.npy  (1000, 617)
  saved datasets/isolet\X_0002.npy  (1000, 617)
  saved datasets/isolet\X_0003.npy  (1000, 617)
  saved datasets/isolet\X_0004.npy  (1000, 617)
  saved datasets/isolet\X_0005.npy  (1000, 617)
  saved datasets/isolet\X_0006.npy  (1000, 617)
  saved datasets/isolet\X_0007.npy  (797, 617)
Written 8 chunk files.
